# Mini Project 1: Word Guessing Game

**PGR211 Advanced Programming for Data Science** — mandatory coursework requirement, Week 38.

## The game

- A secret word is chosen at random from a list.
- The player enters a name, then guesses one letter at a time.
- Correct letters are revealed in their correct positions.
- A wrong letter costs one of the 12 turns; the turn count is configurable.
- The round ends when the word is complete or the turns run out.

## How the design covers Learning Outcome 1

| Concept | Where it appears |
| --- | --- |
| Class and object | `WordGuessingGame`, `ConsoleGameRunner` and the word sources are classes; each round is an object |
| Method | `guess`, `masked_word`, `reveal_word`, `run` |
| Inheritance | `FixedWordSource`, `RandomWordSource` and `DifficultyWordSource` derive from `WordSource`; `DifficultyWordSource` extends `RandomWordSource` via `super()` |
| Polymorphism | `WordGuessingGame` accepts any `WordSource` and calls `next_word()` without knowing the concrete type |
| Encapsulation | The secret word, guessed letters and turn counter are private; state is read through properties and changed only by `guess` |

The rules are kept separate from the console input and output, which is what makes the unit tests at the end of this notebook possible.

## 1. Imports and configuration

In [1]:
from __future__ import annotations

import random
from abc import ABC, abstractmethod
from collections.abc import Callable, Iterable
from enum import Enum, auto

DEFAULT_WORDS: tuple[str, ...] = (
    "kristiania",
    "computer",
    "science",
    "programming",
    "python",
    "inheritance",
    "polymorphism",
    "encapsulation",
)

DEFAULT_MAX_WRONG_GUESSES = 12

## 2. Word sources: inheritance and polymorphism

`WordSource` is an abstract base class. It cannot be instantiated and it defines the single operation the game needs: `next_word()`. Shared validation lives in the base class so no subclass has to repeat it.

Three subclasses implement the abstraction differently:

- `FixedWordSource` returns words in a known order, which makes tests deterministic.
- `RandomWordSource` picks at random, and accepts an injected random generator so the randomness can be seeded.
- `DifficultyWordSource` inherits from `RandomWordSource` and narrows the candidates by word length, delegating the rest of the work to `super().__init__`.

This is polymorphism in the useful sense: the game works with all three and knows about none of them.

In [2]:
class WordSource(ABC):
    """Supplies secret words to a :class:`WordGuessingGame`.

    Subclasses decide the selection strategy. The game only depends on this
    abstract type, so any source can be substituted without changing the rules.
    """

    @abstractmethod
    def next_word(self) -> str:
        """Return the next secret word in normalised (lowercase) form."""

    @staticmethod
    def normalise(word: str) -> str:
        """Trim and lowercase a word, rejecting anything that is not alphabetic.

        Raises:
            ValueError: If the word is empty or contains non-letter characters.
        """
        cleaned = word.strip().lower()
        if not cleaned:
            raise ValueError("Words must not be empty.")
        if not cleaned.isalpha():
            raise ValueError(f"Words must contain letters only, got {word!r}.")
        return cleaned

    @classmethod
    def _normalise_all(cls, words: Iterable[str]) -> list[str]:
        normalised = [cls.normalise(word) for word in words]
        if not normalised:
            raise ValueError("At least one word is required.")
        return normalised


class FixedWordSource(WordSource):
    """Cycles through a fixed sequence of words.

    Deterministic by design, which makes it the source of choice for tests and
    for reproducible demonstrations.
    """

    def __init__(self, words: Iterable[str]) -> None:
        self._words = self._normalise_all(words)
        self._index = 0

    def next_word(self) -> str:
        word = self._words[self._index % len(self._words)]
        self._index += 1
        return word


class RandomWordSource(WordSource):
    """Picks a word uniformly at random.

    Args:
        words: Candidate words.
        rng: Optional random generator. Injecting a seeded ``random.Random``
            makes the selection reproducible in tests.
    """

    def __init__(self, words: Iterable[str], rng: random.Random | None = None) -> None:
        self._words = self._normalise_all(words)
        self._rng = rng if rng is not None else random.Random()

    @property
    def words(self) -> tuple[str, ...]:
        """The candidate words, exposed read-only."""
        return tuple(self._words)

    def next_word(self) -> str:
        return self._rng.choice(self._words)


class DifficultyWordSource(RandomWordSource):
    """A random source restricted to words of a given length range.

    Shorter words are easier to guess, so word length is used here as a simple
    proxy for difficulty.
    """

    def __init__(
        self,
        words: Iterable[str],
        min_length: int = 1,
        max_length: int | None = None,
        rng: random.Random | None = None,
    ) -> None:
        if min_length < 1:
            raise ValueError("min_length must be at least 1.")
        if max_length is not None and max_length < min_length:
            raise ValueError("max_length must not be smaller than min_length.")

        upper_bound = max_length if max_length is not None else float("inf")
        candidates = [
            word
            for word in self._normalise_all(words)
            if min_length <= len(word) <= upper_bound
        ]
        if not candidates:
            raise ValueError(
                f"No words between {min_length} and {max_length} letters long."
            )
        super().__init__(candidates, rng=rng)

## 3. Value types

`GuessOutcome` and `GameState` replace the loose strings and flags a procedural version would use. An enum makes the set of possibilities explicit, and `GuessOutcome.costs_turn` keeps the scoring rule attached to the outcome rather than scattered through the game loop.

In [3]:
class GuessOutcome(Enum):
    """The result of a single guess."""

    CORRECT = auto()
    INCORRECT = auto()
    ALREADY_GUESSED = auto()
    INVALID = auto()

    @property
    def costs_turn(self) -> bool:
        """Whether this outcome consumes one of the player's turns."""
        return self is GuessOutcome.INCORRECT


class GameState(Enum):
    """Where the game currently stands."""

    IN_PROGRESS = auto()
    WON = auto()
    LOST = auto()

## 4. The game: encapsulation

`WordGuessingGame` owns every piece of mutable state. The attributes are private, reads go through read-only properties, and `guess()` is the only way to change anything, so the object cannot be pushed into an inconsistent state from outside.

Two rules are enforced rather than merely documented:

- `reveal_word()` raises if the round is still running, so the interface layer cannot leak the answer by accident.
- `guess()` raises once the round is over, instead of quietly accepting guesses that can no longer matter.

Invalid and repeated guesses are reported back to the caller but cost the player nothing.

In [4]:
class WordGuessingGame:
    """The rules and state of one round of the game.

    The secret word, the guessed letters and the turn counter are private. All
    reads go through read-only properties, and the only way to change state is
    :meth:`guess`, so the game can never be driven into an inconsistent state
    from the outside.

    Args:
        word_source: Supplies the secret word for this round.
        max_wrong_guesses: How many wrong guesses the player may make.
    """

    def __init__(
        self,
        word_source: WordSource,
        max_wrong_guesses: int = DEFAULT_MAX_WRONG_GUESSES,
    ) -> None:
        if max_wrong_guesses < 1:
            raise ValueError("max_wrong_guesses must be at least 1.")
        self._secret_word = word_source.next_word()
        self._max_wrong_guesses = max_wrong_guesses
        self._wrong_guesses = 0
        self._guessed_letters: set[str] = set()

    @property
    def word_length(self) -> int:
        """Number of letters in the secret word."""
        return len(self._secret_word)

    @property
    def remaining_turns(self) -> int:
        """Wrong guesses still available before the round is lost."""
        return self._max_wrong_guesses - self._wrong_guesses

    @property
    def guessed_letters(self) -> frozenset[str]:
        """Every letter guessed so far, as an immutable set."""
        return frozenset(self._guessed_letters)

    @property
    def wrong_letters(self) -> tuple[str, ...]:
        """Guessed letters that are not in the secret word, alphabetically."""
        return tuple(
            sorted(
                letter
                for letter in self._guessed_letters
                if letter not in self._secret_word
            )
        )

    @property
    def state(self) -> GameState:
        """Whether the round is won, lost or still running."""
        if self._is_word_complete():
            return GameState.WON
        if self.remaining_turns <= 0:
            return GameState.LOST
        return GameState.IN_PROGRESS

    @property
    def is_over(self) -> bool:
        """True once the round has been won or lost."""
        return self.state is not GameState.IN_PROGRESS

    def masked_word(self, placeholder: str = "_", separator: str = " ") -> str:
        """Render the secret word with unguessed letters hidden.

        Example:
            ``"s c i e _ c e"`` once every letter but ``n`` has been guessed.
        """
        return separator.join(
            letter if letter in self._guessed_letters else placeholder
            for letter in self._secret_word
        )

    def guess(self, letter: str) -> GuessOutcome:
        """Register a guess and report the outcome.

        Only a single alphabetic character counts as a guess. Invalid and
        repeated guesses are reported back but cost the player nothing.

        Raises:
            RuntimeError: If the round is already over.
        """
        if self.is_over:
            raise RuntimeError("The round is already over; start a new game.")

        candidate = letter.strip().lower()
        if len(candidate) != 1 or not candidate.isalpha():
            return GuessOutcome.INVALID
        if candidate in self._guessed_letters:
            return GuessOutcome.ALREADY_GUESSED

        self._guessed_letters.add(candidate)
        if candidate in self._secret_word:
            return GuessOutcome.CORRECT

        self._wrong_guesses += 1
        return GuessOutcome.INCORRECT

    def reveal_word(self) -> str:
        """Return the secret word, but only after the round has finished.

        Raises:
            RuntimeError: If the round is still in progress.
        """
        if not self.is_over:
            raise RuntimeError("The word can only be revealed once the round is over.")
        return self._secret_word

    def _is_word_complete(self) -> bool:
        return all(letter in self._guessed_letters for letter in self._secret_word)

## 5. Console interface

`ConsoleGameRunner` is the only part of the program that talks to the user. Input and output are injected as functions, defaulting to `input` and `print`, which is what allows the tests below to play whole rounds with scripted keystrokes and assert on what was printed.

In [5]:
class ConsoleGameRunner:
    """Plays a :class:`WordGuessingGame` through a text interface.

    Input and output are injected rather than hard-coded, so the whole loop can
    be exercised in a unit test with scripted keystrokes.

    Args:
        game: The round to play.
        player_name: Name used in the prompts.
        input_fn: Called with a prompt, returns the player's raw input.
        output_fn: Called with each line of output.
    """

    _MESSAGES: dict[GuessOutcome, str] = {
        GuessOutcome.CORRECT: "Correct!",
        GuessOutcome.INCORRECT: "Wrong!",
        GuessOutcome.ALREADY_GUESSED: "You already guessed that letter.",
        GuessOutcome.INVALID: "Please enter exactly one letter.",
    }

    def __init__(
        self,
        game: WordGuessingGame,
        player_name: str = "Player",
        input_fn: Callable[[str], str] = input,
        output_fn: Callable[[str], None] = print,
    ) -> None:
        self._game = game
        self._player_name = player_name.strip() or "Player"
        self._input_fn = input_fn
        self._output_fn = output_fn

    def run(self) -> GameState:
        """Play the round to completion and return the final state."""
        self._announce_start()
        while not self._game.is_over:
            self._show_status()
            guess = self._input_fn("Guess a letter: ")
            outcome = self._game.guess(guess)
            self._output_fn(self._MESSAGES[outcome])
        self._announce_result()
        return self._game.state

    def _announce_start(self) -> None:
        self._output_fn(f"\nGood luck, {self._player_name}!")
        self._output_fn(
            f"The word has {self._game.word_length} letters. "
            "Guess it one letter at a time."
        )

    def _show_status(self) -> None:
        self._output_fn(f"\nWord: {self._game.masked_word()}")
        self._output_fn(f"Turns remaining: {self._game.remaining_turns}")
        wrong_letters = self._game.wrong_letters
        if wrong_letters:
            self._output_fn(f"Wrong letters: {', '.join(wrong_letters)}")

    def _announce_result(self) -> None:
        word = self._game.reveal_word()
        if self._game.state is GameState.WON:
            self._output_fn(f"\nYou win, {self._player_name}! The word was '{word}'.")
        else:
            self._output_fn(f"\nYou lose, {self._player_name}. The word was '{word}'.")

## 6. Entry point

`main()` wires the pieces together. The interactive call is behind a flag so that re-running this notebook top to bottom never blocks on input.

In [6]:
def main() -> None:
    """Prompt for a name and play one round with a random word."""
    try:
        name = input("Enter your name: ")
    except (EOFError, KeyboardInterrupt):
        print("\nNo input received; exiting.")
        return

    game = WordGuessingGame(RandomWordSource(DEFAULT_WORDS))
    runner = ConsoleGameRunner(game, player_name=name)

    try:
        runner.run()
    except (EOFError, KeyboardInterrupt):
        print("\nGame interrupted. Goodbye.")


# Set to True and re-run this cell to play a full interactive round.
PLAY_INTERACTIVELY = False

if PLAY_INTERACTIVELY:
    main()

## 7. Demonstration

A seeded random source and scripted guesses give a reproducible round, so the output below is the same on every run.

In [7]:
scripted_guesses = iter(["a", "e", "i", "o", "u", "s", "c", "n"])

demo_source = RandomWordSource(["science"], rng=random.Random(1))
demo_game = WordGuessingGame(demo_source, max_wrong_guesses=12)
demo_runner = ConsoleGameRunner(
    demo_game,
    player_name="Yorbe",
    input_fn=lambda prompt: next(scripted_guesses),
)

demo_runner.run()


Good luck, Yorbe!
The word has 7 letters. Guess it one letter at a time.

Word: _ _ _ _ _ _ _
Turns remaining: 12
Wrong!

Word: _ _ _ _ _ _ _
Turns remaining: 11
Wrong letters: a
Correct!

Word: _ _ _ e _ _ e
Turns remaining: 11
Wrong letters: a
Correct!

Word: _ _ i e _ _ e
Turns remaining: 11
Wrong letters: a
Wrong!

Word: _ _ i e _ _ e
Turns remaining: 10
Wrong letters: a, o
Wrong!

Word: _ _ i e _ _ e
Turns remaining: 9
Wrong letters: a, o, u
Correct!

Word: s _ i e _ _ e
Turns remaining: 9
Wrong letters: a, o, u
Correct!

Word: s c i e _ c e
Turns remaining: 9
Wrong letters: a, o, u
Correct!

You win, Yorbe! The word was 'science'.


<GameState.WON: 2>

The same game object can be driven by a different source without changing a line of the rules — this is the polymorphism claim, checked rather than asserted.

In [8]:
for source in (
    FixedWordSource(["python"]),
    RandomWordSource(["python"]),
    DifficultyWordSource(["python", "at"], min_length=3),
):
    round_ = WordGuessingGame(source, max_wrong_guesses=1)
    round_.guess("x")
    print(f"{type(source).__name__:>21}: {round_.reveal_word()}")

      FixedWordSource: python
     RandomWordSource: python
 DifficultyWordSource: python


## 8. Unit tests

The tests cover word validation, the three sources, masking, every guess outcome, the win and lose transitions, the encapsulation guarantees and the console loop. They use only the standard library, so the notebook runs anywhere Python 3.10 or newer is installed.

In [9]:
import random
import unittest



def make_game(word: str, max_wrong_guesses: int = 12) -> WordGuessingGame:
    """Build a game with a known secret word."""
    return WordGuessingGame(
        FixedWordSource([word]), max_wrong_guesses=max_wrong_guesses
    )


class ScriptedInput:
    """Replays a fixed list of answers in place of :func:`input`."""

    def __init__(self, answers: list[str]) -> None:
        self._answers = list(answers)
        self.prompts: list[str] = []

    def __call__(self, prompt: str = "") -> str:
        self.prompts.append(prompt)
        if not self._answers:
            raise AssertionError("The runner asked for more input than expected.")
        return self._answers.pop(0)


class WordSourceNormalisationTests(unittest.TestCase):
    def test_words_are_trimmed_and_lowercased(self) -> None:
        source = FixedWordSource(["  Python  "])
        self.assertEqual(source.next_word(), "python")

    def test_empty_word_is_rejected(self) -> None:
        with self.assertRaises(ValueError):
            FixedWordSource(["   "])

    def test_non_alphabetic_word_is_rejected(self) -> None:
        with self.assertRaises(ValueError):
            FixedWordSource(["hello world"])

    def test_empty_word_list_is_rejected(self) -> None:
        with self.assertRaises(ValueError):
            FixedWordSource([])


class FixedWordSourceTests(unittest.TestCase):
    def test_words_are_returned_in_order_then_cycled(self) -> None:
        source = FixedWordSource(["alpha", "beta"])
        self.assertEqual(
            [source.next_word() for _ in range(3)], ["alpha", "beta", "alpha"]
        )


class RandomWordSourceTests(unittest.TestCase):
    def test_selection_is_reproducible_with_a_seeded_generator(self) -> None:
        words = ["alpha", "beta", "gamma", "delta"]
        first = RandomWordSource(words, rng=random.Random(42))
        second = RandomWordSource(words, rng=random.Random(42))
        self.assertEqual(
            [first.next_word() for _ in range(5)],
            [second.next_word() for _ in range(5)],
        )

    def test_selected_word_comes_from_the_candidate_list(self) -> None:
        source = RandomWordSource(["alpha", "beta"])
        self.assertIn(source.next_word(), source.words)


class DifficultyWordSourceTests(unittest.TestCase):
    def test_only_words_within_the_length_range_are_kept(self) -> None:
        source = DifficultyWordSource(
            ["at", "python", "encapsulation"], min_length=3, max_length=8
        )
        self.assertEqual(source.words, ("python",))

    def test_empty_result_is_rejected(self) -> None:
        with self.assertRaises(ValueError):
            DifficultyWordSource(["python"], min_length=20)

    def test_invalid_range_is_rejected(self) -> None:
        with self.assertRaises(ValueError):
            DifficultyWordSource(["python"], min_length=5, max_length=3)


class MaskedWordTests(unittest.TestCase):
    def test_all_letters_are_hidden_before_any_guess(self) -> None:
        game = make_game("science")
        self.assertEqual(game.masked_word(), "_ _ _ _ _ _ _")

    def test_a_correct_guess_reveals_every_occurrence(self) -> None:
        game = make_game("science")
        game.guess("c")
        self.assertEqual(game.masked_word(), "_ c _ _ _ c _")

    def test_placeholder_and_separator_are_configurable(self) -> None:
        game = make_game("cat")
        self.assertEqual(game.masked_word(placeholder="*", separator=""), "***")


class GuessOutcomeTests(unittest.TestCase):
    def test_correct_guess_does_not_cost_a_turn(self) -> None:
        game = make_game("python", max_wrong_guesses=3)
        self.assertIs(game.guess("p"), GuessOutcome.CORRECT)
        self.assertEqual(game.remaining_turns, 3)

    def test_wrong_guess_costs_one_turn(self) -> None:
        game = make_game("python", max_wrong_guesses=3)
        self.assertIs(game.guess("z"), GuessOutcome.INCORRECT)
        self.assertEqual(game.remaining_turns, 2)

    def test_repeated_guess_is_free(self) -> None:
        game = make_game("python", max_wrong_guesses=3)
        game.guess("z")
        self.assertIs(game.guess("z"), GuessOutcome.ALREADY_GUESSED)
        self.assertEqual(game.remaining_turns, 2)

    def test_multi_character_input_is_invalid(self) -> None:
        game = make_game("python", max_wrong_guesses=3)
        self.assertIs(game.guess("py"), GuessOutcome.INVALID)
        self.assertEqual(game.remaining_turns, 3)

    def test_digit_input_is_invalid(self) -> None:
        game = make_game("python", max_wrong_guesses=3)
        self.assertIs(game.guess("4"), GuessOutcome.INVALID)

    def test_invalid_input_is_not_recorded(self) -> None:
        game = make_game("python")
        game.guess("!")
        self.assertEqual(game.guessed_letters, frozenset())

    def test_guesses_are_case_insensitive_and_trimmed(self) -> None:
        game = make_game("python")
        self.assertIs(game.guess("  P  "), GuessOutcome.CORRECT)

    def test_only_incorrect_outcomes_cost_a_turn(self) -> None:
        costly = {outcome for outcome in GuessOutcome if outcome.costs_turn}
        self.assertEqual(costly, {GuessOutcome.INCORRECT})


class GameStateTests(unittest.TestCase):
    def test_game_starts_in_progress(self) -> None:
        self.assertIs(make_game("cat").state, GameState.IN_PROGRESS)

    def test_guessing_every_letter_wins(self) -> None:
        game = make_game("cat")
        for letter in "cat":
            game.guess(letter)
        self.assertIs(game.state, GameState.WON)
        self.assertTrue(game.is_over)

    def test_running_out_of_turns_loses(self) -> None:
        game = make_game("cat", max_wrong_guesses=2)
        game.guess("x")
        game.guess("y")
        self.assertIs(game.state, GameState.LOST)

    def test_win_is_possible_on_the_last_turn(self) -> None:
        game = make_game("cat", max_wrong_guesses=1)
        game.guess("c")
        game.guess("a")
        game.guess("t")
        self.assertIs(game.state, GameState.WON)

    def test_guessing_after_the_round_is_over_raises(self) -> None:
        game = make_game("cat", max_wrong_guesses=1)
        game.guess("x")
        with self.assertRaises(RuntimeError):
            game.guess("c")


class EncapsulationTests(unittest.TestCase):
    def test_word_cannot_be_revealed_while_the_round_runs(self) -> None:
        with self.assertRaises(RuntimeError):
            make_game("cat").reveal_word()

    def test_word_is_revealed_once_the_round_ends(self) -> None:
        game = make_game("cat", max_wrong_guesses=1)
        game.guess("x")
        self.assertEqual(game.reveal_word(), "cat")

    def test_guessed_letters_cannot_be_mutated_from_outside(self) -> None:
        game = make_game("cat")
        game.guess("c")
        self.assertIsInstance(game.guessed_letters, frozenset)
        self.assertEqual(game.guessed_letters, {"c"})

    def test_wrong_letters_exclude_correct_guesses(self) -> None:
        game = make_game("cat")
        game.guess("c")
        game.guess("z")
        game.guess("b")
        self.assertEqual(game.wrong_letters, ("b", "z"))

    def test_invalid_turn_limit_is_rejected(self) -> None:
        with self.assertRaises(ValueError):
            WordGuessingGame(FixedWordSource(["cat"]), max_wrong_guesses=0)


class PolymorphismTests(unittest.TestCase):
    def test_the_game_accepts_any_word_source(self) -> None:
        sources = [
            FixedWordSource(["python"]),
            RandomWordSource(["python"]),
            DifficultyWordSource(["python"], min_length=3),
        ]
        for source in sources:
            with self.subTest(source=type(source).__name__):
                game = WordGuessingGame(source, max_wrong_guesses=1)
                game.guess("x")
                self.assertEqual(game.reveal_word(), "python")


class ConsoleGameRunnerTests(unittest.TestCase):
    def test_a_winning_round_reports_the_word(self) -> None:
        game = make_game("cat")
        output: list[str] = []
        runner = ConsoleGameRunner(
            game,
            player_name="Yorbe",
            input_fn=ScriptedInput(["c", "a", "t"]),
            output_fn=output.append,
        )
        self.assertIs(runner.run(), GameState.WON)
        self.assertIn("You win, Yorbe! The word was 'cat'.", "\n".join(output))

    def test_a_losing_round_reports_the_word(self) -> None:
        game = make_game("cat", max_wrong_guesses=1)
        output: list[str] = []
        runner = ConsoleGameRunner(
            game, input_fn=ScriptedInput(["x"]), output_fn=output.append
        )
        self.assertIs(runner.run(), GameState.LOST)
        self.assertIn("The word was 'cat'.", "\n".join(output))

    def test_invalid_input_is_reported_without_ending_the_round(self) -> None:
        game = make_game("cat")
        output: list[str] = []
        runner = ConsoleGameRunner(
            game, input_fn=ScriptedInput(["12", "c", "a", "t"]), output_fn=output.append
        )
        runner.run()
        joined = "\n".join(output)
        self.assertIn("Please enter exactly one letter.", joined)
        self.assertIn("You win", joined)

    def test_blank_player_name_falls_back_to_default(self) -> None:
        game = make_game("cat", max_wrong_guesses=1)
        output: list[str] = []
        runner = ConsoleGameRunner(
            game,
            player_name="   ",
            input_fn=ScriptedInput(["x"]),
            output_fn=output.append,
        )
        runner.run()
        self.assertIn("Good luck, Player!", "\n".join(output))

    def test_wrong_letters_are_shown_back_to_the_player(self) -> None:
        game = make_game("cat")
        output: list[str] = []
        runner = ConsoleGameRunner(
            game, input_fn=ScriptedInput(["z", "c", "a", "t"]), output_fn=output.append
        )
        runner.run()
        self.assertIn("Wrong letters: z", "\n".join(output))

In [10]:
suite = unittest.TestLoader().loadTestsFromModule(
    __import__("__main__")
)
unittest.TextTestRunner(verbosity=2).run(suite)

test_a_losing_round_reports_the_word (__main__.ConsoleGameRunnerTests.test_a_losing_round_reports_the_word) ... 

ok


test_a_winning_round_reports_the_word (__main__.ConsoleGameRunnerTests.test_a_winning_round_reports_the_word) ... 

ok


test_blank_player_name_falls_back_to_default (__main__.ConsoleGameRunnerTests.test_blank_player_name_falls_back_to_default) ... 

...

ok


test_words_are_trimmed_and_lowercased (__main__.WordSourceNormalisationTests.test_words_are_trimmed_and_lowercased) ... 

ok


----------------------------------------------------------------------
Ran 37 tests in 0.055s

OK


<unittest.runner.TextTestResult run=37 errors=0 failures=0>